In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/processed/cars_preprocessed.csv")

In [2]:
df.shape

(752944, 32)

In [3]:
model_df = df[df["price"] <= 500000].copy()

model_df.shape

(752891, 32)

In [4]:
features = [
    "manufacturer",
    "model",
    "year",
    "mileage",
    "engine_liters",
    "has_accident",
    "is_one_owner",
    "is_personal_use",
    "fuel_type",
    "transmission_type",
    "transmission_gears"
]

model_data = model_df[features + ["price"]].copy()

model_data.shape

(752891, 12)

In [5]:
model_data.isna().sum().sort_values(ascending=False)

transmission_gears    241217
engine_liters          52963
is_one_owner           30761
is_personal_use        24292
has_accident           23656
fuel_type              22149
mileage                  480
manufacturer               0
year                       0
model                      0
transmission_type          0
price                      0
dtype: int64

In [6]:
numeric_cols = [
    "mileage",
    "engine_liters",
    "transmission_gears",
    "has_accident",
    "is_one_owner",
    "is_personal_use"
]

for col in numeric_cols:
    model_data[col] = model_data[col].fillna(model_data[col].median())

model_data["fuel_type"] = model_data["fuel_type"].fillna("Unknown")

In [7]:
model_data.isna().sum()

manufacturer          0
model                 0
year                  0
mileage               0
engine_liters         0
has_accident          0
is_one_owner          0
is_personal_use       0
fuel_type             0
transmission_type     0
transmission_gears    0
price                 0
dtype: int64

In [8]:
X = model_data.drop("price", axis=1)
y = model_data["price"]

X.shape, y.shape

((752891, 11), (752891,))

In [9]:
features = [
    "manufacturer",
    "year",
    "mileage",
    "engine_liters",
    "has_accident",
    "is_one_owner",
    "is_personal_use",
    "fuel_type",
    "transmission_type",
    "transmission_gears"
]

X = model_data[features]
y = model_data["price"]

In [10]:
X.shape

(752891, 10)

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

X_train.shape, X_test.shape

((602312, 10), (150579, 10))

In [12]:
import sys
print(sys.executable)

c:\Users\Sanskar\AppData\Local\Programs\Python\Python313\python.exe


In [13]:


import sklearn
print(sklearn.__version__)

1.9.1


In [14]:
from sklearn.model_selection import train_test_split

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

X_train.shape, X_test.shape

((602312, 10), (150579, 10))

In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_cols = [
    "manufacturer",
    "fuel_type",
    "transmission_type"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ],
    remainder="passthrough"
)

In [17]:
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

In [18]:
X_train_encoded.shape, X_test_encoded.shape

((602312, 76), (150579, 76))

In [19]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

In [20]:
model.fit(X_train_encoded, y_train)


KeyboardInterrupt: 

In [21]:
model = RandomForestRegressor(
    n_estimators=30,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

In [22]:
X_train_small = X_train_encoded[:200000]
y_train_small = y_train.iloc[:200000]

In [23]:
model.fit(X_train_small, y_train_small)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",30
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_

In [24]:
y_pred = model.predict(X_test_encoded)

In [25]:
y_pred[:10]

array([ 12722.01840648,  11139.40515188, 137329.83980159,  35853.30191532,
        20558.31668803,  17056.49629203,  28118.11757278,  39147.66056797,
        29251.03303406,  27947.91625251])

In [26]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test, y_pred)

print("Mean Absolute Error:", round(mae, 2))

Mean Absolute Error: 4845.73


In [ ]:
y_pred[:10]

array([ 12722.01840648,  11139.40515188, 137329.83980159,  35853.30191532,
        20558.31668803,  17056.49629203,  28118.11757278,  39147.66056797,
        29251.03303406,  27947.91625251])

In [27]:
baseline_price = y_train.median()

baseline_pred = [baseline_price] * len(y_test)

baseline_mae = mean_absolute_error(y_test, baseline_pred)

print("Baseline MAE:", round(baseline_mae, 2))
print("Random Forest MAE:", round(mae, 2))

Baseline MAE: 13679.95
Random Forest MAE: 4845.73


In [28]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, y_pred)

print("R² Score:", round(r2, 3))

R² Score: 0.801


In [29]:
baseline_results = {
    "model": "Random Forest - Baseline",
    "MAE": mae,
    "R2": r2
}

baseline_results

{'model': 'Random Forest - Baseline',
 'MAE': 4845.725967137974,
 'R2': 0.8006387594763388}

In [30]:
model_data["model"].nunique()

12177

In [31]:
model_data["model"].value_counts().head(20)

model
Fusion SE              3172
Sportage LX            2847
Corolla LE             2834
GLC 300 Base 4MATIC    2667
Sentra SV              2636
Explorer XLT           2542
Rogue SV               2524
Optima LX              2502
Tundra SR5             2455
Sorento LX             2433
Odyssey EX-L           2352
Forte LXS              2318
Wrangler Sport         2305
Focus SE               2285
F-150 XLT              2277
RX 350 Base            2275
Edge SEL               2269
Escape SE              2210
Renegade Latitude      2174
Encore Preferred       2094
Name: count, dtype: int64

In [32]:
# Keep the 500 most common car models
top_models = model_data["model"].value_counts().head(500).index

model_data["model_group"] = model_data["model"].where(
    model_data["model"].isin(top_models),
    "Other"
)

model_data["model_group"].nunique()

501

In [33]:
model_data["model_group"].value_counts().head(10)

model_group
Other                  323511
Fusion SE                3172
Sportage LX              2847
Corolla LE               2834
GLC 300 Base 4MATIC      2667
Sentra SV                2636
Explorer XLT             2542
Rogue SV                 2524
Optima LX                2502
Tundra SR5               2455
Name: count, dtype: int64

In [34]:
features = [
    "manufacturer",
    "model_group",
    "year",
    "mileage",
    "engine_liters",
    "has_accident",
    "is_one_owner",
    "is_personal_use",
    "fuel_type",
    "transmission_type",
    "transmission_gears"
]

X = model_data[features]
y = model_data["price"]

In [35]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [36]:
categorical_cols = [
    "manufacturer",
    "model_group",
    "fuel_type",
    "transmission_type"
]

In [37]:
preprocessor_model = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ],
    remainder="passthrough"
)

X_train_encoded_model = preprocessor_model.fit_transform(X_train)
X_test_encoded_model = preprocessor_model.transform(X_test)

print(X_train_encoded_model.shape)
print(X_test_encoded_model.shape)

(602312, 577)
(150579, 577)


In [38]:
model_2 = RandomForestRegressor(
    n_estimators=30,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

X_train_small = X_train_encoded_model[:200000]
y_train_small = y_train.iloc[:200000]

model_2.fit(X_train_small, y_train_small)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",30
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_

In [39]:
y_pred_2 = model_2.predict(X_test_encoded_model)

In [40]:
from sklearn.metrics import mean_absolute_error, r2_score

mae_2 = mean_absolute_error(y_test, y_pred_2)
r2_2 = r2_score(y_test, y_pred_2)

print(f"Model 2 MAE: ${mae_2:,.2f}")
print(f"Model 2 R²: {r2_2:.3f}")

Model 2 MAE: $4,545.95
Model 2 R²: 0.817


In [41]:
model_data["vehicle_age"] = 2024 - model_data["year"]

In [42]:

model_data["vehicle_age"].describe()

count    752891.000000
mean          6.209372
std           5.113164
min           0.000000
25%           3.000000
50%           5.000000
75%           8.000000
max         109.000000
Name: vehicle_age, dtype: float64

In [43]:
features = [
    "manufacturer",
    "model_group",
    "vehicle_age",
    "mileage",
    "engine_liters",
    "has_accident",
    "is_one_owner",
    "is_personal_use",
    "fuel_type",
    "transmission_type",
    "transmission_gears"
]

In [44]:
X = model_data[features]
y = model_data["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [45]:
categorical_cols = [
    "manufacturer",
    "model_group",
    "fuel_type",
    "transmission_type"
]

preprocessor_model3 = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ],
    remainder="passthrough"
)

X_train_encoded_3 = preprocessor_model3.fit_transform(X_train)
X_test_encoded_3 = preprocessor_model3.transform(X_test)

print(X_train_encoded_3.shape)
print(X_test_encoded_3.shape)

(602312, 577)
(150579, 577)


In [46]:
model_3 = RandomForestRegressor(
    n_estimators=30,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

X_train_small_3 = X_train_encoded_3[:200000]
y_train_small_3 = y_train.iloc[:200000]

model_3.fit(X_train_small_3, y_train_small_3)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",30
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_

In [47]:
y_pred_3 = model_3.predict(X_test_encoded_3)

In [48]:
mae_3 = mean_absolute_error(y_test, y_pred_3)
r2_3 = r2_score(y_test, y_pred_3)

print(f"Model 3 MAE: ${mae_3:,.2f}")
print(f"Model 3 R²: {r2_3:.3f}")

Model 3 MAE: $4,546.04
Model 3 R²: 0.817


In [49]:
results = X_test.copy()

results["actual_price"] = y_test.values
results["predicted_fair_price"] = y_pred_2

results.head()

,manufacturer,model_group,vehicle_age,mileage,engine_liters,has_accident,is_one_owner,is_personal_use,fuel_type,transmission_type,transmission_gears,actual_price,predicted_fair_price
335563,Hyundai,Santa Fe Sport 2.4L,8,129280.0,2.4,1.0,0.0,0.0,Gasoline,Automatic,6.0,13999.0,12852.634135
83739,Cadillac,Other,10,171000.0,3.6,0.0,1.0,1.0,Gasoline,Other,7.0,17500.0,11128.271056
601467,Porsche,Other,8,21500.0,3.8,0.0,1.0,1.0,Gasoline,Other,6.0,105000.0,136251.634893
500336,Mazda,Other,2,5478.0,2.5,0.0,0.0,0.0,Gasoline,Automatic,6.0,29397.0,35276.070383
329241,Honda,CR-V EX,6,95655.0,1.5,0.0,1.0,1.0,Gasoline,CVT,7.0,22488.0,20700.933645


In [50]:
results["price_difference"] = (
    results["actual_price"] - results["predicted_fair_price"]
)

In [51]:
results["price_difference_pct"] = (
    results["price_difference"]
    / results["predicted_fair_price"]
) * 100

In [52]:
results[
    [
        "actual_price",
        "predicted_fair_price",
        "price_difference",
        "price_difference_pct"
    ]
].head(10)

,actual_price,predicted_fair_price,price_difference,price_difference_pct
335563,13999.0,12852.634135,1146.365865,8.919307
83739,17500.0,11128.271056,6371.728944,57.257133
601467,105000.0,136251.634893,-31251.634893,-22.936705
500336,29397.0,35276.070383,-5879.070383,-16.665888
329241,22488.0,20700.933645,1787.066355,8.632781
317065,15997.0,17555.900722,-1558.900722,-8.879640
535218,31990.0,29782.146361,2207.853639,7.413346
91675,38900.0,39447.338097,-547.338097,-1.387516
248681,23653.0,29251.033034,-5598.033034,-19.137899
11096,31931.0,27980.355524,3950.644476,14.119351


In [53]:
results["price_difference_pct"].describe()

count    150579.000000
mean         -0.197968
std          22.025213
min         -99.996581
25%         -11.255764
50%          -1.426240
75%           8.603601
max        1149.344408
Name: price_difference_pct, dtype: float64

In [54]:
results["price_difference_pct"].quantile(
    [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
)

0.05   -28.386961
0.10   -21.449092
0.25   -11.255764
0.50    -1.426240
0.75     8.603601
0.90    20.744525
0.95    30.529803
Name: price_difference_pct, dtype: float64

In [55]:
results["deal_status"] = results["price_difference_pct"].apply(
    lambda x:
        "Potentially Undervalued" if x <= -10
        else "Potentially Overpriced" if x > 10
        else "Fair"
)

In [56]:
results["deal_status"].value_counts()

deal_status
Fair                       75128
Potentially Undervalued    41544
Potentially Overpriced     33907
Name: count, dtype: int64

In [57]:
results.groupby("deal_status")["price_difference_pct"].agg(
    ["count", "mean", "median", "min", "max"]
)

,count,mean,median,min,max
deal_status,,,,,
Fair,75128,-0.415694,-0.565697,-9.999966,9.999608
Potentially Overpriced,33907,25.766084,19.174515,10.000779,1149.344408
Potentially Undervalued,41544,-20.995333,-18.113720,-99.996581,-10.000405


In [58]:
deal_results = results[
    [
        "manufacturer",
        "model_group",
        "vehicle_age",
        "mileage",
        "actual_price",
        "predicted_fair_price",
        "price_difference",
        "price_difference_pct",
        "deal_status"
    ]
].copy()

In [59]:
top_deals = deal_results[
    deal_results["deal_status"] == "Potentially Undervalued"
].sort_values(
    "price_difference_pct"
).head(10)

top_deals

,manufacturer,model_group,vehicle_age,mileage,actual_price,predicted_fair_price,price_difference,price_difference_pct,deal_status
652426,Subaru,Other,5,31570.0,1.0,29251.033034,-29250.033034,-99.996581,Potentially Undervalued
13572,Audi,Other,10,98881.0,1.0,13263.828856,-13262.828856,-99.992461,Potentially Undervalued
726291,Volkswagen,Other,17,133964.0,1.0,8668.741189,-8667.741189,-99.988464,Potentially Undervalued
29947,BMW,Other,27,1.0,2999.0,134959.324621,-131960.324621,-97.777849,Potentially Undervalued
559979,Nissan,Other,15,45615.0,750.0,13456.250976,-12706.250976,-94.426382,Potentially Undervalued
456214,Land Rover,Other,17,0.0,7988.0,118029.766667,-110041.766667,-93.232216,Potentially Undervalued
246969,Ford,Other,31,1.0,2790.0,39344.467481,-36554.467481,-92.908787,Potentially Undervalued
48703,BMW,Other,8,103601.0,2500.0,29645.966930,-27145.966930,-91.567150,Potentially Undervalued
69585,Buick,Other,16,92272.0,999.0,10668.476498,-9669.476498,-90.635964,Potentially Undervalued
688856,Toyota,Sienna XLE,26,113533.0,5999.0,63264.631990,-57265.631990,-90.517609,Potentially Undervalued


In [60]:
valid_deals = deal_results[
    deal_results["actual_price"] > 1000
].copy()

In [61]:
top_deals = valid_deals[
    valid_deals["deal_status"] == "Potentially Undervalued"
].sort_values(
    "price_difference_pct"
).head(10)

top_deals

,manufacturer,model_group,vehicle_age,mileage,actual_price,predicted_fair_price,price_difference,price_difference_pct,deal_status
29947,BMW,Other,27,1.0,2999.0,134959.324621,-131960.324621,-97.777849,Potentially Undervalued
456214,Land Rover,Other,17,0.0,7988.0,118029.766667,-110041.766667,-93.232216,Potentially Undervalued
246969,Ford,Other,31,1.0,2790.0,39344.467481,-36554.467481,-92.908787,Potentially Undervalued
48703,BMW,Other,8,103601.0,2500.0,29645.966930,-27145.966930,-91.567150,Potentially Undervalued
688856,Toyota,Sienna XLE,26,113533.0,5999.0,63264.631990,-57265.631990,-90.517609,Potentially Undervalued
177476,Dodge,Other,37,324939.0,1299.0,13603.749387,-12304.749387,-90.451162,Potentially Undervalued
601582,Porsche,Other,27,43647.0,14998.0,152942.733333,-137944.733333,-90.193715,Potentially Undervalued
86916,Cadillac,Other,26,20793.0,5600.0,49886.584148,-44286.584148,-88.774537,Potentially Undervalued
664255,Toyota,Other,27,130000.0,4900.0,43346.269640,-38446.269640,-88.695682,Potentially Undervalued
454246,Land Rover,Other,31,103508.0,3700.0,32571.565799,-28871.565799,-88.640399,Potentially Undervalued


In [62]:
credible_deals = deal_results[
    (deal_results["actual_price"] > 5000) &
    (deal_results["predicted_fair_price"] > 5000)
].copy()

In [63]:
top_deals = credible_deals[
    credible_deals["deal_status"] == "Potentially Undervalued"
].sort_values(
    "price_difference_pct"
).head(10)

top_deals

,manufacturer,model_group,vehicle_age,mileage,actual_price,predicted_fair_price,price_difference,price_difference_pct,deal_status
456214,Land Rover,Other,17,0.0,7988.0,118029.766667,-110041.766667,-93.232216,Potentially Undervalued
688856,Toyota,Sienna XLE,26,113533.0,5999.0,63264.631990,-57265.631990,-90.517609,Potentially Undervalued
601582,Porsche,Other,27,43647.0,14998.0,152942.733333,-137944.733333,-90.193715,Potentially Undervalued
86916,Cadillac,Other,26,20793.0,5600.0,49886.584148,-44286.584148,-88.774537,Potentially Undervalued
605591,Porsche,Other,40,4565.0,13900.0,115131.343333,-101231.343333,-87.926832,Potentially Undervalued
673178,Toyota,Other,30,101156.0,8500.0,64600.349519,-56100.349519,-86.842176,Potentially Undervalued
605560,Porsche,Other,42,97555.0,8950.0,65873.400853,-56923.400853,-86.413332,Potentially Undervalued
266178,Ford,Other,56,65200.0,10999.0,77692.421974,-66693.421974,-85.842892,Potentially Undervalued
82960,Cadillac,Other,56,32073.0,6795.0,45991.955907,-39196.955907,-85.225677,Potentially Undervalued
163945,Dodge,Other,13,0.0,6995.0,40967.341396,-33972.341396,-82.925424,Potentially Undervalued


In [64]:
model_data["model"].value_counts().describe()


count    12177.000000
mean        61.828940
std        202.964503
min          1.000000
25%          1.000000
50%          4.000000
75%         26.000000
max       3172.000000
Name: count, dtype: float64

In [65]:
(model_data["model"].value_counts() >= 10).sum()

np.int64(4547)

In [66]:
model_data["mileage_band"] = pd.cut(
    model_data["mileage"],
    bins=[-1, 25000, 50000, 75000, 100000, float("inf")],
    labels=["0-25K", "25-50K", "50-75K", "75-100K", "100K+"]
)

In [67]:
comparable_counts = (
    model_data
    .groupby(
        ["manufacturer", "model", "year", "mileage_band"],
        observed=True
    )["price"]
    .transform("count")
)

comparable_counts.describe()

count    752891.000000
mean         65.627647
std         101.987397
min           1.000000
25%          12.000000
50%          32.000000
75%          81.000000
max        1393.000000
Name: price, dtype: float64

In [68]:
market_fair_price = (
    model_data
    .groupby(
        ["manufacturer", "model", "year", "mileage_band"],
        observed=True
    )["price"]
    .transform("median")
)

model_data["market_fair_price"] = market_fair_price
model_data["comparable_count"] = comparable_counts

In [69]:
model_data[
    ["model", "year", "mileage_band", "market_fair_price", "comparable_count"]
].head(10)

,model,year,mileage_band,market_fair_price,comparable_count
0,ILX Hybrid 1.5L,2013,75-100K,13988.0,1
1,ILX Hybrid 1.5L,2013,25-50K,17995.0,1
2,ILX Hybrid 1.5L,2013,50-75K,16975.0,4
3,ILX Hybrid 1.5L,2013,100K+,14728.0,2
4,ILX Hybrid 1.5L,2013,100K+,14728.0,2
5,ILX Hybrid 1.5L,2013,50-75K,16975.0,4
6,ILX Hybrid 1.5L,2013,50-75K,16975.0,4
7,ILX Hybrid 1.5L,2014,100K+,8500.0,1
8,ILX Hybrid 1.5L,2013,50-75K,16975.0,4
9,NSX,1993,50-75K,84995.0,1


In [70]:
model_data["market_fair_price"] = model_data["market_fair_price"].where(
    model_data["comparable_count"] >= 5
)

In [71]:
model_data["market_fair_price"].notna().value_counts()

market_fair_price
True     666155
False     86736
Name: count, dtype: int64

In [72]:
# Create training benchmark data
train_benchmark = model_data.loc[
    X_train.index,
    ["manufacturer", "model", "year", "mileage_band", "price"]
].copy()

# Calculate median price and number of comparables
benchmark_stats = (
    train_benchmark
    .groupby(
        ["manufacturer", "model", "year", "mileage_band"],
        observed=True
    )["price"]
    .agg(
        market_fair_price="median",
        comparable_count="count"
    )
    .reset_index()
)

benchmark_stats.head()

,manufacturer,model,year,mileage_band,market_fair_price,comparable_count
0,Acura,CL 3.0,1997,25-50K,11937.0,1
1,Acura,CL 3.0,1997,100K+,7950.0,1
2,Acura,CL 3.2,2003,100K+,7439.5,2
3,Acura,CL 3.2 Type S,2001,50-75K,10950.0,1
4,Acura,CL 3.2 Type S,2001,100K+,3995.0,1


In [73]:
# Create test benchmark data
test_benchmark = model_data.loc[
    X_test.index,
    ["manufacturer", "model", "year", "mileage_band"]
].copy()

# Match each test car to its training-market benchmark
test_benchmark = test_benchmark.merge(
    benchmark_stats,
    on=["manufacturer", "model", "year", "mileage_band"],
    how="left"
)

test_benchmark.head()

,manufacturer,model,year,mileage_band,market_fair_price,comparable_count
0,Hyundai,Santa Fe Sport 2.4L,2016,100K+,14165.5,34.0
1,Cadillac,SRX Base,2014,100K+,12443.0,8.0
2,Porsche,Boxster Spyder,2016,0-25K,101415.0,7.0
3,Mazda,CX-5 2.5 S Select Package,2022,0-25K,28990.0,23.0
4,Honda,CR-V EX,2018,75-100K,22992.5,28.0


In [74]:
test_benchmark["market_fair_price"] = test_benchmark[
    "market_fair_price"
].where(
    test_benchmark["comparable_count"] >= 5
)

In [75]:
test_benchmark["market_fair_price"].notna().value_counts()

market_fair_price
True     126324
False     24255
Name: count, dtype: int64

In [76]:
# Combine actual prices, ML predictions, and market benchmarks
hybrid_results = X_test.copy()

hybrid_results["actual_price"] = y_test.values
hybrid_results["ml_fair_price"] = y_pred_2

hybrid_results["market_fair_price"] = test_benchmark[
    "market_fair_price"
].values

hybrid_results["comparable_count"] = test_benchmark[
    "comparable_count"
].values

# Use market price when enough comparables exist,
# otherwise fall back to ML
hybrid_results["final_fair_price"] = hybrid_results[
    "market_fair_price"
].fillna(
    hybrid_results["ml_fair_price"]
)

hybrid_results.head()

,manufacturer,model_group,vehicle_age,mileage,engine_liters,has_accident,is_one_owner,is_personal_use,fuel_type,transmission_type,transmission_gears,actual_price,ml_fair_price,market_fair_price,comparable_count,final_fair_price
335563,Hyundai,Santa Fe Sport 2.4L,8,129280.0,2.4,1.0,0.0,0.0,Gasoline,Automatic,6.0,13999.0,12852.634135,14165.5,34.0,14165.5
83739,Cadillac,Other,10,171000.0,3.6,0.0,1.0,1.0,Gasoline,Other,7.0,17500.0,11128.271056,12443.0,8.0,12443.0
601467,Porsche,Other,8,21500.0,3.8,0.0,1.0,1.0,Gasoline,Other,6.0,105000.0,136251.634893,101415.0,7.0,101415.0
500336,Mazda,Other,2,5478.0,2.5,0.0,0.0,0.0,Gasoline,Automatic,6.0,29397.0,35276.070383,28990.0,23.0,28990.0
329241,Honda,CR-V EX,6,95655.0,1.5,0.0,1.0,1.0,Gasoline,CVT,7.0,22488.0,20700.933645,22992.5,28.0,22992.5


In [77]:
from sklearn.metrics import mean_absolute_error, r2_score

hybrid_mae = mean_absolute_error(
    hybrid_results["actual_price"],
    hybrid_results["final_fair_price"]
)

hybrid_r2 = r2_score(
    hybrid_results["actual_price"],
    hybrid_results["final_fair_price"]
)

print("Hybrid MAE:", round(hybrid_mae, 2))
print("Hybrid R²:", round(hybrid_r2, 3))

Hybrid MAE: 2942.66
Hybrid R²: 0.895


In [78]:
hybrid_results["price_difference"] = (
    hybrid_results["actual_price"]
    - hybrid_results["final_fair_price"]
)

hybrid_results["price_difference_pct"] = (
    hybrid_results["price_difference"]
    / hybrid_results["final_fair_price"]
) * 100

hybrid_results["deal_status"] = hybrid_results[
    "price_difference_pct"
].apply(
    lambda x:
        "Potentially Undervalued" if x <= -10
        else "Potentially Overpriced" if x > 10
        else "Fair"
)

hybrid_results[
    [
        "actual_price",
        "final_fair_price",
        "price_difference_pct",
        "deal_status"
    ]
].head(10)

,actual_price,final_fair_price,price_difference_pct,deal_status
335563,13999.0,14165.500000,-1.175391,Fair
83739,17500.0,12443.000000,40.641324,Potentially Overpriced
601467,105000.0,101415.000000,3.534980,Fair
500336,29397.0,28990.000000,1.403932,Fair
329241,22488.0,22992.500000,-2.194194,Fair
317065,15997.0,17555.900722,-8.879640,Fair
535218,31990.0,32196.000000,-0.639831,Fair
91675,38900.0,39447.338097,-1.387516,Fair
248681,23653.0,22555.000000,4.868100,Fair
11096,31931.0,34998.000000,-8.763358,Fair


In [79]:
deal_summary = (
    hybrid_results["deal_status"]
    .value_counts()
    .rename_axis("deal_status")
    .reset_index(name="count")
)

deal_summary["percentage"] = (
    deal_summary["count"]
    / len(hybrid_results)
    * 100
).round(2)

deal_summary

,deal_status,count,percentage
0,Fair,102517,68.08
1,Potentially Overpriced,24757,16.44
2,Potentially Undervalued,23305,15.48


In [80]:
credible_deals = hybrid_results[
    (hybrid_results["actual_price"] > 5000) &
    (hybrid_results["final_fair_price"] > 5000) &
    (hybrid_results["deal_status"] == "Potentially Undervalued")
].copy()

credible_deals.shape

(22397, 19)

In [81]:
top_deals = credible_deals.sort_values(
    "price_difference"
).head(20)

top_deals[
    [
        "manufacturer",
        "model_group",
        "vehicle_age",
        "mileage",
        "actual_price",
        "final_fair_price",
        "price_difference",
        "price_difference_pct",
        "comparable_count"
    ]
]

,manufacturer,model_group,vehicle_age,mileage,actual_price,final_fair_price,price_difference,price_difference_pct,comparable_count
510875,Mercedes-Benz,Other,3,28.0,139655.0,417444.000000,-277789.000000,-66.545213,12.0
520823,Mercedes-Benz,Other,4,724.0,134999.0,357887.244444,-222888.244444,-62.278901,2.0
177794,Dodge,Other,14,2350.0,97000.0,296373.105972,-199373.105972,-67.270984,1.0
341341,Hyundai,Other,1,12.0,37250.0,202388.583972,-165138.583972,-81.594812,1.0
411510,Kia,Other,1,15.0,56000.0,202388.583972,-146388.583972,-72.330455,4.0
601582,Porsche,Other,27,43647.0,14998.0,152942.733333,-137944.733333,-90.193715,NaN
534921,Mercedes-Benz,Other,3,34.0,79295.0,208342.000000,-129047.000000,-61.939983,11.0
533434,Mercedes-Benz,Other,11,7925.0,174545.0,299609.755556,-125064.755556,-41.742551,2.0
166787,Dodge,Other,11,10800.0,128995.0,241978.816667,-112983.816667,-46.691615,2.0
606824,Porsche,Other,29,33221.0,60975.0,172222.100000,-111247.100000,-64.595136,1.0


In [82]:
def get_confidence(count):
    if pd.isna(count):
        return "ML Fallback"
    elif count >= 20:
        return "Strong"
    elif count >= 10:
        return "Moderate"
    elif count >= 5:
        return "Basic"
    else:
        return "Insufficient"

hybrid_results["deal_confidence"] = (
    hybrid_results["comparable_count"]
    .apply(get_confidence)
)

hybrid_results[
    [
        "actual_price",
        "final_fair_price",
        "price_difference_pct",
        "deal_status",
        "comparable_count",
        "deal_confidence"
    ]
].head(10)

,actual_price,final_fair_price,price_difference_pct,deal_status,comparable_count,deal_confidence
335563,13999.0,14165.500000,-1.175391,Fair,34.0,Strong
83739,17500.0,12443.000000,40.641324,Potentially Overpriced,8.0,Basic
601467,105000.0,101415.000000,3.534980,Fair,7.0,Basic
500336,29397.0,28990.000000,1.403932,Fair,23.0,Strong
329241,22488.0,22992.500000,-2.194194,Fair,28.0,Strong
317065,15997.0,17555.900722,-8.879640,Fair,3.0,Insufficient
535218,31990.0,32196.000000,-0.639831,Fair,24.0,Strong
91675,38900.0,39447.338097,-1.387516,Fair,NaN,ML Fallback
248681,23653.0,22555.000000,4.868100,Fair,207.0,Strong
11096,31931.0,34998.000000,-8.763358,Fair,15.0,Moderate


In [83]:
hybrid_results["deal_confidence"].value_counts()

deal_confidence
Strong          84355
Moderate        25066
Insufficient    17504
Basic           16903
ML Fallback      6751
Name: count, dtype: int64

In [84]:
credible_deals = hybrid_results[
    (hybrid_results["deal_status"] == "Potentially Undervalued") &
    (hybrid_results["comparable_count"] >= 5) &
    (hybrid_results["actual_price"] > 5000)
].copy()

print("Credible potential deals:", len(credible_deals))

Credible potential deals: 14548


In [85]:
top_percentage_deals = credible_deals.sort_values(
    "price_difference_pct"
).head(15)

top_percentage_deals[
    [
        "manufacturer",
        "model_group",
        "vehicle_age",
        "mileage",
        "actual_price",
        "final_fair_price",
        "price_difference_pct",
        "comparable_count",
        "deal_confidence"
    ]
]

,manufacturer,model_group,vehicle_age,mileage,actual_price,final_fair_price,price_difference_pct,comparable_count,deal_confidence
630454,Subaru,Ascent Touring 7-Passenger,3,2630.0,9795.0,40269.0,-75.676078,65.0,Strong
670855,Toyota,Venza LE,3,5634.0,9450.0,36854.0,-74.358279,72.0,Strong
628647,RAM,ProMaster 2500 High Roof,6,58426.0,9795.0,31995.0,-69.385842,34.0,Strong
222808,Ford,Other,22,41923.0,5795.0,18888.0,-69.319144,11.0,Moderate
404873,Jeep,Renegade Limited,8,69031.0,5995.0,18988.0,-68.427428,23.0,Strong
324733,Honda,Other,5,46831.0,8950.0,27999.0,-68.034573,70.0,Strong
214760,Ford,Other,12,210404.0,7100.0,21950.0,-67.653759,9.0,Basic
277213,GMC,Other,3,8992.0,13795.0,41995.0,-67.150851,31.0,Strong
253481,Ford,Transit-150 Base,5,59000.0,11795.0,35889.5,-67.135235,28.0,Strong
99533,Chevrolet,Other,21,84175.0,6995.0,20992.5,-66.678576,6.0,Basic


In [86]:
deal = top_percentage_deals.iloc[0]

manufacturer = deal["manufacturer"]
model = deal["model_group"]
year = 2024 - deal["vehicle_age"]
mileage = deal["mileage"]

print("Manufacturer:", manufacturer)
print("Model:", model)
print("Year:", year)
print("Mileage:", mileage)

Manufacturer: Subaru
Model: Ascent Touring 7-Passenger
Year: 2021
Mileage: 2630.0


In [87]:
comparables = model_data[
    (model_data["manufacturer"] == manufacturer) &
    (model_data["model_group"] == model) &
    (model_data["year"] == year) &
    (model_data["mileage_band"] == pd.cut(
        [mileage],
        bins=[-1, 25000, 50000, 75000, 100000, float("inf")],
        labels=["0-25K", "25-50K", "50-75K", "75-100K", "100K+"]
    )[0])
]

comparables[
    [
        "manufacturer",
        "model",
        "year",
        "mileage",
        "price",
        "has_accident",
        "is_one_owner"
    ]
].sort_values("price").head(20)

,manufacturer,model,year,mileage,price,has_accident,is_one_owner
630799,Subaru,Ascent Touring 7-Passenger,2021,4314.0,8995.0,1.0,1.0
630454,Subaru,Ascent Touring 7-Passenger,2021,2630.0,9795.0,1.0,1.0
630416,Subaru,Ascent Touring 7-Passenger,2021,22000.0,24000.0,1.0,1.0
632148,Subaru,Ascent Touring 7-Passenger,2021,8264.0,30991.0,1.0,1.0
630933,Subaru,Ascent Touring 7-Passenger,2021,9011.0,31000.0,1.0,0.0
631438,Subaru,Ascent Touring 7-Passenger,2021,12749.0,31999.0,1.0,1.0
631347,Subaru,Ascent Touring 7-Passenger,2021,20332.0,34400.0,1.0,1.0
632155,Subaru,Ascent Touring 7-Passenger,2021,21400.0,34447.0,1.0,1.0
630923,Subaru,Ascent Touring 7-Passenger,2021,7695.0,34498.0,1.0,1.0
630878,Subaru,Ascent Touring 7-Passenger,2021,11555.0,34600.0,0.0,1.0


In [88]:
accident_adjusted_comparables = model_data[
    (model_data["manufacturer"] == manufacturer) &
    (model_data["model_group"] == model) &
    (model_data["year"] == year) &
    (model_data["mileage_band"] == comparables["mileage_band"].iloc[0]) &
    (model_data["has_accident"] == deal["has_accident"])
]

accident_adjusted_comparables[
    ["model", "mileage", "price", "has_accident"]
].sort_values("price").head(20)

,model,mileage,price,has_accident
630799,Ascent Touring 7-Passenger,4314.0,8995.0,1.0
630454,Ascent Touring 7-Passenger,2630.0,9795.0,1.0
630416,Ascent Touring 7-Passenger,22000.0,24000.0,1.0
632148,Ascent Touring 7-Passenger,8264.0,30991.0,1.0
630933,Ascent Touring 7-Passenger,9011.0,31000.0,1.0
631438,Ascent Touring 7-Passenger,12749.0,31999.0,1.0
631347,Ascent Touring 7-Passenger,20332.0,34400.0,1.0
632155,Ascent Touring 7-Passenger,21400.0,34447.0,1.0
630923,Ascent Touring 7-Passenger,7695.0,34498.0,1.0
631445,Ascent Touring 7-Passenger,16741.0,35000.0,1.0


In [89]:
accident_status = hybrid_results.loc[
    deal.name, "has_accident"
]

print(accident_status)

1.0


In [90]:
deal["has_accident"]

np.float64(1.0)

In [91]:
accident_status

np.float64(1.0)

In [92]:
accident_status = 1.0

accident_adjusted_comparables = model_data[
    (model_data["manufacturer"] == manufacturer) &
    (model_data["model_group"] == model) &
    (model_data["year"] == year) &
    (model_data["mileage_band"] == comparables["mileage_band"].iloc[0]) &
    (model_data["has_accident"] == accident_status)
]

accident_adjusted_comparables[
    ["model", "mileage", "price", "has_accident"]
].sort_values("price").head(20)

,model,mileage,price,has_accident
630799,Ascent Touring 7-Passenger,4314.0,8995.0,1.0
630454,Ascent Touring 7-Passenger,2630.0,9795.0,1.0
630416,Ascent Touring 7-Passenger,22000.0,24000.0,1.0
632148,Ascent Touring 7-Passenger,8264.0,30991.0,1.0
630933,Ascent Touring 7-Passenger,9011.0,31000.0,1.0
631438,Ascent Touring 7-Passenger,12749.0,31999.0,1.0
631347,Ascent Touring 7-Passenger,20332.0,34400.0,1.0
632155,Ascent Touring 7-Passenger,21400.0,34447.0,1.0
630923,Ascent Touring 7-Passenger,7695.0,34498.0,1.0
631445,Ascent Touring 7-Passenger,16741.0,35000.0,1.0


In [93]:
train_condition_benchmark = model_data.loc[
    X_train.index,
    [
        "manufacturer",
        "model",
        "year",
        "mileage_band",
        "has_accident",
        "price"
    ]
].copy()

condition_benchmark_stats = (
    train_condition_benchmark
    .groupby(
        [
            "manufacturer",
            "model",
            "year",
            "mileage_band",
            "has_accident"
        ],
        observed=True
    )["price"]
    .agg(
        market_fair_price="median",
        comparable_count="count"
    )
    .reset_index()
)

condition_benchmark_stats.head()

,manufacturer,model,year,mileage_band,has_accident,market_fair_price,comparable_count
0,Acura,CL 3.0,1997,25-50K,0.0,11937.0,1
1,Acura,CL 3.0,1997,100K+,0.0,7950.0,1
2,Acura,CL 3.2,2003,100K+,0.0,6899.0,1
3,Acura,CL 3.2,2003,100K+,1.0,7980.0,1
4,Acura,CL 3.2 Type S,2001,50-75K,0.0,10950.0,1


In [94]:
test_condition_benchmark = model_data.loc[
    X_test.index,
    [
        "manufacturer",
        "model",
        "year",
        "mileage_band",
        "has_accident"
    ]
].copy()

test_condition_benchmark = test_condition_benchmark.merge(
    condition_benchmark_stats,
    on=[
        "manufacturer",
        "model",
        "year",
        "mileage_band",
        "has_accident"
    ],
    how="left"
)

test_condition_benchmark["market_fair_price"] = (
    test_condition_benchmark["market_fair_price"].where(
        test_condition_benchmark["comparable_count"] >= 5
    )
)

test_condition_benchmark["market_fair_price"].notna().value_counts()

market_fair_price
True     112258
False     38321
Name: count, dtype: int64

In [95]:
condition_hybrid = hybrid_results.copy()

condition_hybrid["condition_market_price"] = (
    test_condition_benchmark["market_fair_price"].values
)

condition_hybrid["condition_comparable_count"] = (
    test_condition_benchmark["comparable_count"].values
)

condition_hybrid["condition_fair_price"] = (
    condition_hybrid["condition_market_price"]
    .fillna(condition_hybrid["ml_fair_price"])
)

In [96]:
condition_mae = mean_absolute_error(
    condition_hybrid["actual_price"],
    condition_hybrid["condition_fair_price"]
)

condition_r2 = r2_score(
    condition_hybrid["actual_price"],
    condition_hybrid["condition_fair_price"]
)

print("Condition-aware Hybrid MAE:", round(condition_mae, 2))
print("Condition-aware Hybrid R²:", round(condition_r2, 3))

Condition-aware Hybrid MAE: 3096.75
Condition-aware Hybrid R²: 0.889


In [97]:
final_deal_results = hybrid_results[
    [
        "manufacturer",
        "model_group",
        "vehicle_age",
        "mileage",
        "actual_price",
        "final_fair_price",
        "price_difference",
        "price_difference_pct",
        "deal_status",
        "comparable_count",
        "deal_confidence"
    ]
].copy()

final_deal_results.head()

,manufacturer,model_group,vehicle_age,mileage,actual_price,final_fair_price,price_difference,price_difference_pct,deal_status,comparable_count,deal_confidence
335563,Hyundai,Santa Fe Sport 2.4L,8,129280.0,13999.0,14165.5,-166.5,-1.175391,Fair,34.0,Strong
83739,Cadillac,Other,10,171000.0,17500.0,12443.0,5057.0,40.641324,Potentially Overpriced,8.0,Basic
601467,Porsche,Other,8,21500.0,105000.0,101415.0,3585.0,3.534980,Fair,7.0,Basic
500336,Mazda,Other,2,5478.0,29397.0,28990.0,407.0,1.403932,Fair,23.0,Strong
329241,Honda,CR-V EX,6,95655.0,22488.0,22992.5,-504.5,-2.194194,Fair,28.0,Strong


In [98]:
final_deal_results.to_csv(
    "../data/processed/fair_price_results.csv",
    index=False
)

print("Saved successfully!")
print(final_deal_results.shape)

Saved successfully!
(150579, 11)


In [99]:
import os

file_path = "../data/processed/fair_price_results.csv"

print("File exists:", os.path.exists(file_path))
print("File size (MB):", round(os.path.getsize(file_path) / (1024 * 1024), 2))

File exists: True
File size (MB): 14.08
